In [0]:
# Inject WHL to bypass Serverless environment dependencies crash


# **1.Dependencies & Imports**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# **2.Widgets (Parameters)**

In [0]:
# Day3.2 module bootstrap (direct-silver)

import sys
from pathlib import Path

_local_src = str(Path.cwd() / "src")
if _local_src not in sys.path:
    sys.path.insert(0, _local_src)

from lakehouse.crypto_parser import interval_to_timedelta

print("[day3.2] imported lakehouse modules for silver pipeline")

In [0]:
%skip
dbutils.widgets.text("catalog", "enterprise")
dbutils.widgets.text("bronze_schema", "bronze_market")
dbutils.widgets.text("silver_schema", "silver_market")

dbutils.widgets.text("bronze_table", "crypto_ohlc_raw")
dbutils.widgets.text("silver_table", "crypto_ohlc_1d")
dbutils.widgets.text("silver_table", "")
dbutils.widgets.text("interval", "1d")

INTERVAL = dbutils.widgets.get("interval").strip()
SILVER_TABLE_IN = dbutils.widgets.get("silver_table").strip()

if not SILVER_TABLE_IN:
    SILVER_TABLE = f"crypto_ohlc_{INTERVAL}"  # 1d -> crypto_ohlc_1d
else:
    SILVER_TABLE = SILVER_TABLE_IN

dbutils.widgets.text("source", "coinbase")  # filter source (optional)
dbutils.widgets.text("symbols", "")  # comma-separated, optional
dbutils.widgets.text("start_date", "")  # YYYY-MM-DD, optional (event_time filter)
dbutils.widgets.text("end_date", "")  # YYYY-MM-DD, optional (event_time filter)

CATALOG = dbutils.widgets.get("catalog").strip()
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema").strip()
SILVER_SCHEMA = dbutils.widgets.get("silver_schema").strip()
BRONZE_TABLE = dbutils.widgets.get("bronze_table").strip()
SILVER_TABLE = dbutils.widgets.get("silver_table").strip()

SOURCE = dbutils.widgets.get("source").strip()
SYMBOLS_RAW = dbutils.widgets.get("symbols").strip()
SYMBOLS = [s.strip().upper() for s in SYMBOLS_RAW.split(",") if s.strip()] if SYMBOLS_RAW else []

START_DATE = dbutils.widgets.get("start_date").strip()
END_DATE = dbutils.widgets.get("end_date").strip()

BRONZE_TBL = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"
SILVER_TBL = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"

print(f"[INFO] Bronze: {BRONZE_TBL}")
print(f"[INFO] Silver: {SILVER_TBL}")

In [0]:
# 2.Widgets (Parameters)
dbutils.widgets.text("catalog", "enterprise")
dbutils.widgets.text("bronze_schema", "bronze_market")
dbutils.widgets.text("silver_schema", "silver_market")

dbutils.widgets.text("bronze_table", "crypto_ohlc_raw")

# 新增：interval（决定 Silver 输出粒度与默认表名）
dbutils.widgets.text("interval", "1d")  # 1d / 1m / 1h

# silver_table 可留空：留空则自动 crypto_ohlc_{interval}
dbutils.widgets.text("silver_table", "")

dbutils.widgets.text("source", "coinbase")  # optional
dbutils.widgets.text("symbols", "")  # optional
dbutils.widgets.text("start_date", "")  # optional
dbutils.widgets.text("end_date", "")  # optional

CATALOG = dbutils.widgets.get("catalog").strip()
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema").strip()
SILVER_SCHEMA = dbutils.widgets.get("silver_schema").strip()
BRONZE_TABLE = dbutils.widgets.get("bronze_table").strip()

INTERVAL = dbutils.widgets.get("interval").strip()
SILVER_TABLE_IN = dbutils.widgets.get("silver_table").strip()

SILVER_TABLE = SILVER_TABLE_IN if SILVER_TABLE_IN else f"crypto_ohlc_{INTERVAL}"

SOURCE = dbutils.widgets.get("source").strip()
SYMBOLS_RAW = dbutils.widgets.get("symbols").strip()
SYMBOLS = [s.strip().upper() for s in SYMBOLS_RAW.split(",") if s.strip()] if SYMBOLS_RAW else []

START_DATE = dbutils.widgets.get("start_date").strip()
END_DATE = dbutils.widgets.get("end_date").strip()

BRONZE_TBL = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"
SILVER_TBL = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"

print(f"[INFO] Bronze: {BRONZE_TBL}")
print(f"[INFO] Silver: {SILVER_TBL}")
print(f"[INFO] interval={INTERVAL}, source={SOURCE}, symbols={SYMBOLS}")

# **3.Read Bronze with filters**

In [0]:
bronze = spark.table(BRONZE_TBL)
# optional filters
if SOURCE:
    bronze = bronze.where(F.col("source") == F.lit(SOURCE))

if SYMBOLS:
    bronze = bronze.where(F.col("symbol").isin(SYMBOLS))

# enforce interval scope to avoid cross-interval contamination
if "interval" in bronze.columns:
    bronze = bronze.where(F.col("interval") == F.lit(INTERVAL))

# event_time filter: uses event_time (day anchor) from Bronze
if START_DATE:
    bronze = bronze.where(F.col("event_time") >= F.to_timestamp(F.lit(f"{START_DATE} 00:00:00")))
if END_DATE:
    bronze = bronze.where(F.col("event_time") <= F.to_timestamp(F.lit(f"{END_DATE} 23:59:59")))

# **4.Parse raw_json into "klines" array**

In [0]:
# Reuse the filtered bronze DataFrame from the previous cell; avoid redundant full table read
print("[INFO] Reusing filtered bronze DataFrame from previous cell.")

In [0]:
payload_schema = T.StructType(
    [
        T.StructField("symbol", T.StringType(), True),
        T.StructField("interval", T.StringType(), True),
        T.StructField("klines", T.ArrayType(T.ArrayType(T.DoubleType())), True),
    ]
)

# 解析 JSON
parsed = bronze.withColumn("payload", F.from_json(F.col("raw_json"), payload_schema)).withColumn(
    "klines", F.col("payload.klines")
)

# 炸裂数组 (Explode)
k = (
    parsed.select(
        F.col("source"),
        F.col("symbol"),
        F.col("ingestion_ts"),
        F.explode_outer("klines").alias("kline"),
    )
    # Coinbase 数组长度固定为 6，过滤掉任何不完整的数据
    .filter(F.size(F.col("kline")) >= 6)
)

# interval-aware bar end calculation
interval_sec = int(interval_to_timedelta(INTERVAL).total_seconds())

# 解析列 (专为 Coinbase 定制)
# Coinbase 官方 API 顺序: [0:Time, 1:Low, 2:High, 3:Open, 4:Close, 5:Volume]
k2 = (
    k
    # 时间戳兼容秒/毫秒
    .withColumn("_ts_raw", F.col("kline").getItem(0).cast("double"))
    .withColumn(
        "_ts_sec",
        F.when(F.col("_ts_raw") > F.lit(10_000_000_000), F.col("_ts_raw") / 1000.0)
        .otherwise(F.col("_ts_raw"))
        .cast("long"),
    )
    .withColumn("bar_start_ts", F.from_unixtime(F.col("_ts_sec")).cast("timestamp"))
    # OHLCV 数值映射
    .withColumn("low", F.col("kline").getItem(1).cast("double"))
    .withColumn("high", F.col("kline").getItem(2).cast("double"))
    .withColumn("open", F.col("kline").getItem(3).cast("double"))
    .withColumn("close", F.col("kline").getItem(4).cast("double"))
    .withColumn("volume", F.col("kline").getItem(5).cast("double"))
    # 结束时间根据 interval 动态计算
    .withColumn(
        "bar_end_ts", F.from_unixtime(F.col("_ts_sec") + F.lit(interval_sec)).cast("timestamp")
    )
    .drop("_ts_raw", "_ts_sec")
)

# 最终选择列
silver_df = k2.withColumn("p_date", F.to_date("bar_start_ts")).select(
    "source",
    "symbol",
    "bar_start_ts",
    "bar_end_ts",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "ingestion_ts",
    "p_date",
)

# **5. Data quality filters (basic, safe)**


In [0]:
silver_df = (
    silver_df.where(F.col("bar_start_ts").isNotNull())
    .where(F.col("open") > 0)
    .where(F.col("high") > 0)
    .where(F.col("low") > 0)
    .where(F.col("close") > 0)
    .where(F.col("volume") >= 0)
    .where(F.col("high") >= F.greatest(F.col("open"), F.col("close")))
    .where(F.col("low") <= F.least(F.col("open"), F.col("close")))
)

# intra-batch dedupe
silver_df = silver_df.dropDuplicates(["source", "symbol", "bar_start_ts"])

print("[INFO] Parsed rows (after quality+dedupe) ready for merge.")

# **6.Ensure Silver table exists (if not created yet)**

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TBL} (
  source        STRING,
  symbol        STRING,
  interval      STRING, 
  bar_start_ts  TIMESTAMP,
  bar_end_ts    TIMESTAMP,
  open          DOUBLE,
  high          DOUBLE,
  low           DOUBLE,
  close         DOUBLE,
  volume        DOUBLE,
  ingestion_ts  TIMESTAMP,
  p_date        DATE
)
USING DELTA
PARTITIONED BY (p_date)
""")

# 7. MERGE into Silver (dedupe by source+symbol+bar_start_ts)

In [0]:
# 7. BATCH WRITE into Silver (MERGE INTO for deduplication)
silver_df = silver_df.withColumn("interval", F.lit(INTERVAL))

silver_df.createOrReplaceTempView("updates")
spark.sql(f"""
MERGE INTO {SILVER_TBL} t
USING updates s
ON t.source = s.source
AND t.symbol = s.symbol
AND t.interval = s.interval
AND t.bar_start_ts = s.bar_start_ts
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print("[INFO] Batch MERGE into Silver completed.")

# **8.Sanity checks**
